# Modern Makine Öğrenmesi Mimarisi Geliştirme
## Hafta 2 — Problem Tanımından Modellemeye Hazır Veriye

> **Ana fikir:** İyi bir ML projesi algoritmayla değil, doğru kararın ve doğru verinin tanımlanmasıyla başlar.

**Ders süresi:** 60 dk anlatım + 60 dk uygulama  
**Geçen haftadan bağlantı:** Notebook'ta keşfettiğimiz mantığı modüler ve tekrar üretilebilir bir projeye taşımıştık. Bu hafta henüz model kurmadan önce problemi ve veriyi doğru hazırlayacağız.

### Bu dersin sonunda ulaşmak istediğim kazanımlar
Bu dersin sonunda birlikte:
- Belirsiz bir iş fikrini ölçülebilir bir ML problemine çevirebileceğiz.
- Hedef değişkeni, gözlem birimini, tahmin anını ve tahmin ufkunu tanımlayabileceğiz.
- Bağımsız değişkenleri yalnızca tahmin anında erişilebilir bilgilerden seçebileceğiz.
- ETL ve ELT yaklaşımlarını karşılaştırabileceğiz.
- Pandas ile Polars arasında ihtiyaca göre seçim yapabileceğiz.
- Feature Store, point-in-time correctness ve train–serve skew kavramlarını açıklayabileceğiz.
- Bir veri setini modellemeden önce sistematik olarak inceleyebileceğiz.

---
### Bugünkü ders akışımız
1. **0–25 dk:** İş fikirlerini ML problemine çevirme
2. **25–40 dk:** Veri mühendisliği, ETL ve ELT
3. **40–50 dk:** Pandas ve Polars
4. **50–60 dk:** Feature Store ve train–serve skew
5. **60–120 dk:** ISP churn projesi için veri üretimi ve veri tartışması


## 1. Açılış: İş fikri ile ML problemi aynı şey değildir

Derse iş birimlerinden gelebilecek şu cümlelerle başlayalım:

- **Pazarlama:** “Kampanyaya doğru müşterileri seçelim.”
- **Müşteri deneyimi:** “Ayrılacak müşterileri önceden bulalım.”
- **Şebeke operasyonu:** “Arızaları oluşmadan tahmin edelim.”
- **Çağrı merkezi:** “Gelecek ay kaç çağrı geleceğini bilelim.”
- **Yatırım:** “Fiber yatırımını doğru mahallelere yapalım.”

Bu cümlelerin hiçbiri henüz tam bir ML problemi değildir. Önce hangi kararı destekleyeceğimizi, neyi tahmin edeceğimizi ve tahmin sonucuyla hangi aksiyonu alacağımızı netleştirmemiz gerekir.

> İlk sorum “Hangi algoritmayı kullanalım?” değil, **“Bu tahminle hangi kararı değiştireceğiz?”** olacak.

## 2. Bir iş fikrini ML problemine çevirmek için kullanacağım çerçeve

Her fikir için şu sekiz soruyu soracağım:

| Soru | Açıklama |
|---|---|
| **Karar** | Model çıktısıyla hangi iş kararını değiştireceğiz? |
| **Gözlem birimi** | Bir satır müşteri mi, cihaz mı, mahalle mi, gün mü? |
| **Tahmin anı** | Skoru tam olarak ne zaman üreteceğiz? |
| **Hedef** | Gelecekte gerçekleşen hangi olayı öğreneceğiz? |
| **Tahmin ufku** | Sonraki 7 gün mü, 30 gün mü, 6 ay mı? |
| **Özellikler** | Tahmin anında gerçekten hangi bilgilere sahibiz? |
| **Aksiyon** | Yüksek/düşük skor sonrasında ne yapacağız? |
| **Başarı ölçütü** | Model metriğinin yanında iş başarısını nasıl ölçeceğiz? |

Bu tanımlar yapılmadan oluşturulan hedef değişken kolayca veri sızıntısı, yanlış optimizasyon veya aksiyona dönüşmeyen bir skor üretebilir.

## 3. Farklı iş fikirlerini birlikte tartışalım

| İş fikri | Problem tipi | Gözlem birimi | Olası hedef | Olası özellikler | Model sonrası aksiyon |
|---|---|---|---|---|---|
| Kampanya seçimi | Sınıflandırma / uplift | Müşteri–kampanya | 14 günde kampanyaya dönüş | Geçmiş kullanım, kanal etkileşimi, segment | Teklif gönder / gönderme |
| Churn önleme | Sınıflandırma | Müşteri–snapshot | 30 günde ayrılma | Kullanım trendi, arıza, şikâyet, kontrat | Retention aksiyonu |
| Çağrı hacmi | Zaman serisi / regresyon | Gün veya saat | Gelecek gün çağrı sayısı | Takvim, kampanya, arıza, geçmiş hacim | Vardiya planlama |
| Şebeke arızası | Sınıflandırma / survival | Cihaz–snapshot | 7 günde arıza | Alarm, sıcaklık, trafik, bakım geçmişi | Önleyici bakım |
| Fiber yatırım | Ranking / regresyon | Mahalle–dönem | 6 ayda yeni abone veya gelir | Demografi, mesafe, talep, rekabet | Yatırım önceliği |

### Sınıf tartışması
Her satır için şunları sorgulayalım:
1. Hedefi ne zaman gözlemleyebiliriz?
2. Özellikler tahmin anında mevcut mu?
3. Yanlış pozitif ve yanlış negatif kararın maliyeti nedir?
4. Model olmadan kullanılan basit bir iş kuralı var mı?
5. Tahmin üretilse bile iş birimi aksiyon alabilir mi?

## 4. Aynı fikir, farklı hedef değişkenler

“Müşteri kaybını tahmin edelim” dediğimizde bile birden fazla hedef tanımı mümkündür:

- Sonraki 30 günde aboneliğini iptal etti mi?
- Sonraki 60 günde başka operatöre geçti mi?
- Önümüzdeki ay kullanımını %80'den fazla azalttı mı?
- Sonraki 90 günde ödeme nedeniyle pasife düştü mü?

Bunların her biri farklı örneklem, farklı veri, farklı aksiyon ve farklı model üretir.

### Bugün kullanacağım hedef tanımı
> Ay sonu itibarıyla aktif olan bir müşterinin **sonraki 30 gün içinde hizmetini iptal etmesi**.

- **Gözlem birimi:** `customer_id + snapshot_date`
- **Tahmin anı:** Her ayın son günü
- **Hedef:** `churn_30d`
- **Aksiyon:** Uygun müşteriye retention teklifi veya proaktif destek
- **İş metriği:** Önlenen churn kaynaklı net gelir / aksiyon maliyeti


## 5. Bağımsız değişkenleri seçerken en önemli kural

> Bir özellik, yalnızca **tahmin anında erişilebiliyorsa** modele girebilir.

| Kullanılabilir özellik | Riskli / sızıntılı özellik |
|---|---|
| Son 30 gündeki kullanım | İptal sonrası kapanış nedeni |
| Son 90 gündeki şikâyet sayısı | İptal talebi açılma tarihi |
| Snapshot günündeki kontrat tipi | Gelecek ay oluşan fatura |
| Snapshot öncesi arıza sayısı | Churn sonrası yapılan anket |
| Mevcut modem yaşı | Hizmet kapanış tarihi |

Bir kolonun teknik olarak tabloda bulunması, tahmin anında kullanılabileceği anlamına gelmez. Kolonların iş anlamını ve oluşma zamanını bilmemiz gerekir.

## 6. Veri mühendisliği neden ML dersinin parçasıdır?

Bir ML sistemi genellikle tek bir temiz CSV ile başlamaz. Churn örneğinde veriler farklı sistemlerden gelebilir:

- CRM: müşteri ve kontrat
- Billing: fatura ve ödeme
- Network: kullanım, hız, kesinti
- Call center: çağrı ve şikâyet
- Campaign: geçmiş teklifler ve dönüşler

Bu kaynakları doğru zamanda, doğru anahtarla ve tekrarlanabilir şekilde bir araya getirmeden iyi bir modelleme tabanı oluşturamayız. Veri mühendisliği burada modelin ön hazırlığı değil, ML sisteminin omurgasıdır.

## 7. ETL ve ELT

### ETL — Extract, Transform, Load
```text
Kaynak sistemler → Dönüştürme → Hedef veri ambarı
```
Veriyi hedefe yüklemeden önce temizler ve dönüştürürüz. Hedef sistemin kısıtlı olduğu veya sıkı kontrol gerektiği durumlarda anlamlıdır.

### ELT — Extract, Load, Transform
```text
Kaynak sistemler → Ham veri katmanı → Hedef sistemde dönüştürme
```
Önce ham veriyi veri gölüne/ambarına alır, dönüşümü güçlü hedef sistemde yaparız. Modern bulut veri platformlarında sık görülür.

| Boyut | ETL | ELT |
|---|---|---|
| Dönüşüm zamanı | Yüklemeden önce | Yükledikten sonra |
| Ham veriyi koruma | Her zaman değil | Genellikle evet |
| Hesaplama | Ayrı işlem katmanı | Hedef platform |
| Yeniden üretim | Kaynak yeniden gerekebilir | Ham katmandan yapılabilir |
| Tipik kullanım | Klasik ambar / regüle akış | Lakehouse / cloud warehouse |

Burada tek bir doğru yoktur. Veri hacmi, güvenlik, maliyet, gecikme ve yönetişim gereksinimine göre seçim yaparım.

## 8. Ham veriden modelleme tablosuna katmanlar

```text
Kaynaklar
   ↓
Raw / Bronze     → Kaynağa en yakın, değiştirilmeyen veri
   ↓
Clean / Silver   → Tipler, anahtarlar, eksikler, tekrarlar düzeltilmiş
   ↓
Feature / Gold   → İş tanımlı, zaman uyumlu model özellikleri
   ↓
Training table   → X + hedef + snapshot bilgisi
```

Her katmanda şu sorulara cevap ararım:
- Veri nereden geldi?
- Hangi dönüşümler uygulandı?
- Hangi tarih/snapshot için geçerli?
- Tekrar çalıştırınca aynı sonucu üretiyor mu?
- Kalite kontrolleri başarısız olursa akış duruyor mu?

## 9. Pandas ne zaman yeterli, Polars ne zaman anlamlı?

Pandas yetersiz veya kötü bir araç değildir. Küçük–orta ölçekli veri, keşif ve zengin ekosistem entegrasyonu için güçlü bir varsayılandır. Sorun, veri ve işlem büyüdüğünde aynı yaklaşımı sorgulamadan sürdürmektir.

| İhtiyaç | Pandas | Polars |
|---|---|---|
| Öğrenme ve ekosistem | Çok güçlü | Gelişiyor |
| Tek makinede keşif | Çok uygun | Çok uygun |
| Çok çekirdekli yürütme | Sınırlı olabilir | Varsayılan olarak güçlü |
| Lazy execution | Temel kullanımda yok | Güçlü |
| Sorgu optimizasyonu | Sınırlı | Predicate/projection pushdown |
| Bellek verimliliği | Veri tiplerine dikkat ister | Arrow tabanlı ve verimli |

### Karar verirken bakacağım sinyaller
- Veri belleğe sığmıyor veya swap oluşturuyor mu?
- Aynı aggregation dakikalarca sürüyor mu?
- Sadece birkaç kolon gerektiği halde bütün dosyayı mı okuyoruz?
- İşlem zinciri lazy çalıştırmadan fayda görür mü?
- Ekip ve downstream kütüphaneler hangi veri yapısını bekliyor?


### Aynı dönüşümün Pandas ve Polars karşılığı

**Pandas**
```python
result = (
    df.loc[df["is_active"] == 1]
      .groupby("contract_type", as_index=False)
      .agg(avg_fee=("monthly_fee", "mean"), customers=("customer_id", "nunique"))
)
```

**Polars (lazy)**
```python
result = (
    pl.scan_parquet("data/processed/customer_snapshot.parquet")
      .filter(pl.col("is_active") == 1)
      .group_by("contract_type")
      .agg(
          pl.col("monthly_fee").mean().alias("avg_fee"),
          pl.col("customer_id").n_unique().alias("customers"),
      )
      .collect()
)
```

Bu hafta Polars'ın bütün API'sini öğrenmeyeceğiz. Amacım, veri büyüdüğünde araç seçiminin de mimari bir karar olduğunu göstermek.

## 10. Feature Store nedir?

Feature Store'u yalnızca feature'ların tutulduğu bir veritabanı olarak düşünmem. Asıl değer; feature tanımı, hesaplama mantığı, zaman bilgisi, yeniden kullanım ve training–serving tutarlılığıdır.

```text
Kaynak veriler → Ortak feature pipeline
                         ├── Offline store → Training / batch scoring
                         └── Online store  → Düşük gecikmeli serving
```

Bir feature kaydında görmek isteyeceğim bilgiler:
- Adı ve iş tanımı
- Veri tipi ve sahibi
- Entity anahtarı (`customer_id`)
- Event/snapshot zamanı
- Hesaplama penceresi (`son 30 gün`)
- Güncellenme sıklığı
- Null ve geçerlilik kuralları

> Her projede özel bir Feature Store ürünü kurmak zorunda değiliz; fakat feature sözleşmesini ve tekil hesaplama mantığını mutlaka yönetmeliyiz.

## 11. Train–serve skew nedir?

Modeli eğitirken kullandığımız feature ile canlıda hesapladığımız feature aynı adı taşısa bile farklı anlam veya hesaplama kullanıyorsa train–serve skew oluşur.

### Örnek
- **Training:** `avg_usage_30d` = snapshot tarihinden önceki tam 30 gün
- **Serving:** `avg_usage_30d` = içinde bulunulan takvim ayındaki günlerin ortalaması

Ayın 3'ünde serving tarafı yalnız üç günlük veri kullanırken training tarafı 30 günlük veri kullanmış olur. Kolon adı aynı, dağılım ve anlam farklıdır.

### Başka nedenler
- Training'de median, serving'de `0` ile null doldurma
- Farklı kategori sözlükleri
- Saat dilimi farkı
- Training'de gelecekte oluşan kayıtların yanlışlıkla kullanılması
- Batch pipeline ile API kodunun ayrı ayrı yazılması

Önlemek için aynı dönüşüm kodunu paylaşır, feature tanımlarını sürümler, point-in-time join kullanır ve dağılımları iki ortam arasında test ederim.

## 12. Lab projemizi seçelim: ISP churn

Churn projesini seçiyorum çünkü:
- Sınıflandırma problemi olarak anlaşılması kolay.
- Farklı iş sistemlerinden gelen verileri temsil ediyor.
- Zaman, leakage ve class imbalance tartışmasına uygun.
- Bir sonraki hafta LightGBM, validation ve Optuna çalışmalarına doğrudan temel olacak.

Dışarıdan bir veri dosyasına ihtiyaç duymayacağız. Aşağıdaki hücre gerçekçi fakat tamamen sentetik bir müşteri snapshot verisi üretecek. Veri gerçek müşteri bilgisi içermez.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

PROJECT_ROOT = Path.cwd() / "course_workspace" / "isp-churn-project"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

N_CUSTOMERS = 5_000
print(f"Proje dizini hazır: {PROJECT_ROOT}")

In [ ]:
# Bir satır = bir müşterinin ay sonundaki snapshot'ı
snapshot_choices = pd.to_datetime(["2026-01-31", "2026-02-28", "2026-03-31", "2026-04-30"])
snapshot_date = rng.choice(snapshot_choices, size=N_CUSTOMERS)
tenure_months = np.clip(rng.gamma(shape=2.2, scale=15, size=N_CUSTOMERS).round(), 1, 120).astype(int)
contract_type = rng.choice(["monthly", "annual", "two_year"], p=[0.56, 0.31, 0.13], size=N_CUSTOMERS)
city_tier = rng.choice(["metro", "large", "small"], p=[0.45, 0.35, 0.20], size=N_CUSTOMERS)
payment_method = rng.choice(["autopay", "card", "transfer"], p=[0.45, 0.35, 0.20], size=N_CUSTOMERS)
monthly_fee = np.clip(rng.normal(390, 85, N_CUSTOMERS), 150, 750).round(2)
avg_usage_gb_30d = np.clip(rng.gamma(4, 22, N_CUSTOMERS), 2, 400).round(1)
usage_change_90d_pct = np.clip(rng.normal(-2, 22, N_CUSTOMERS), -90, 100).round(1)
support_calls_90d = np.clip(rng.poisson(1.3, N_CUSTOMERS), 0, 10)
network_incidents_30d = np.clip(rng.poisson(0.7, N_CUSTOMERS), 0, 7)
late_payments_6m = np.clip(rng.poisson(0.45, N_CUSTOMERS), 0, 6)
nps_score = np.clip(rng.normal(7.0, 2.0, N_CUSTOMERS).round(), 0, 10)
modem_age_months = np.clip(rng.gamma(2.5, 12, N_CUSTOMERS).round(), 1, 96).astype(int)

# Churn olasılığını iş sezgisine benzeyen ilişkilerle üretiyoruz.
logit = (
    -3.1
    + 0.48 * (contract_type == "monthly")
    - 0.55 * (contract_type == "two_year")
    + 0.24 * support_calls_90d
    + 0.31 * network_incidents_30d
    + 0.28 * late_payments_6m
    - 0.020 * tenure_months
    - 0.16 * nps_score
    - 0.014 * usage_change_90d_pct
    + 0.0020 * (monthly_fee - 350)
)
churn_probability = 1 / (1 + np.exp(-logit))
churn_30d = rng.binomial(1, churn_probability)

raw_df = pd.DataFrame({
    "customer_id": [f"C{i:06d}" for i in range(1, N_CUSTOMERS + 1)],
    "snapshot_date": pd.to_datetime(snapshot_date),
    "tenure_months": tenure_months,
    "contract_type": contract_type,
    "city_tier": city_tier,
    "payment_method": payment_method,
    "monthly_fee": monthly_fee,
    "avg_usage_gb_30d": avg_usage_gb_30d,
    "usage_change_90d_pct": usage_change_90d_pct,
    "support_calls_90d": support_calls_90d,
    "network_incidents_30d": network_incidents_30d,
    "late_payments_6m": late_payments_6m,
    "nps_score": nps_score,
    "modem_age_months": modem_age_months,
    "churn_30d": churn_30d,
})

raw_df.head()

## 13. Gerçek hayata benzetmek için veri kalitesi kusurları ekleyelim

Gerçek bir projede veri genellikle kusursuz gelmez. Eğitim amacıyla:
- Bazı `monthly_fee` ve `nps_score` değerlerini eksik bırakacağım.
- Az sayıda yinelenen satır ekleyeceğim.
- Ham veriyi CSV olarak kaydedeceğim.

Bu sorunları hemen düzeltmek yerine önce tespit edeceğiz; çünkü doğru sıralama **profil çıkarma → karar verme → dönüşüm → doğrulama** olmalıdır.

In [ ]:
missing_fee_idx = rng.choice(raw_df.index, size=75, replace=False)
remaining_idx = raw_df.index.difference(missing_fee_idx)
missing_nps_idx = rng.choice(remaining_idx, size=110, replace=False)
raw_df.loc[missing_fee_idx, "monthly_fee"] = np.nan
raw_df.loc[missing_nps_idx, "nps_score"] = np.nan

duplicate_rows = raw_df.sample(15, random_state=RANDOM_SEED)
raw_df = pd.concat([raw_df, duplicate_rows], ignore_index=True)

RAW_CSV_PATH = RAW_DATA_DIR / "customer_snapshot_raw.csv"
raw_df.to_csv(RAW_CSV_PATH, index=False)

print(f"Ham veri oluşturuldu: {RAW_CSV_PATH}")
print(f"Boyut: {raw_df.shape}")

## 14. Veri sözlüğü

| Kolon | Açıklama | Zaman penceresi | Rol |
|---|---|---|---|
| `customer_id` | Müşteri anahtarı | Snapshot anı | Entity |
| `snapshot_date` | Tahminin üretildiği tarih | — | Zaman anahtarı |
| `tenure_months` | Müşteri kıdemi | Snapshot'a kadar | Feature |
| `contract_type` | Mevcut kontrat türü | Snapshot anı | Feature |
| `city_tier` | Bölge grubu | Snapshot anı | Feature |
| `payment_method` | Güncel ödeme yöntemi | Snapshot anı | Feature |
| `monthly_fee` | Güncel aylık ücret | Snapshot anı | Feature |
| `avg_usage_gb_30d` | Ortalama internet kullanımı | Geçmiş 30 gün | Feature |
| `usage_change_90d_pct` | Kullanım trendi | Geçmiş 90 gün | Feature |
| `support_calls_90d` | Destek çağrısı sayısı | Geçmiş 90 gün | Feature |
| `network_incidents_30d` | Şebeke olayı sayısı | Geçmiş 30 gün | Feature |
| `late_payments_6m` | Gecikmiş ödeme sayısı | Geçmiş 6 ay | Feature |
| `nps_score` | Son bilinen memnuniyet skoru | Snapshot'a kadar | Feature |
| `modem_age_months` | Modem yaşı | Snapshot anı | Feature |
| `churn_30d` | Sonraki 30 günde iptal | Gelecek 30 gün | Target |

Veri sözlüğü yalnız açıklama dokümanı değildir; feature'ların zaman anlamını ve sahipliğini görünür hale getirir.

## 15. İlk bakış: şema ve örnek kayıtlar

Model kurmadan önce şu sırayla ilerleyeceğim:
1. Boyut ve gözlem birimi
2. Veri tipleri
3. Eksik değerler
4. Tekrarlı anahtarlar
5. Hedef dağılımı
6. Sayısal aralıklar ve kategoriler
7. Zaman kapsamı ve leakage kontrolü


In [ ]:
df = pd.read_csv(RAW_CSV_PATH, parse_dates=["snapshot_date"])
print("Boyut:", df.shape)
print("Tarih aralığı:", df["snapshot_date"].min().date(), "→", df["snapshot_date"].max().date())
print("\nİlk üç kayıt:")
print(df.head(3).to_string(index=False))
print("\nVeri tipleri:")
df.dtypes.rename("dtype").to_frame()

In [ ]:
quality_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=False),
}).sort_values(["missing_count", "unique_count"], ascending=[False, True])

quality_report

In [ ]:
duplicate_key = ["customer_id", "snapshot_date"]
duplicate_count = df.duplicated(subset=duplicate_key, keep=False).sum()
exact_duplicate_count = df.duplicated().sum()

print("Tekrarlı entity–snapshot satırı:", duplicate_count)
print("Birebir tekrar eden ek satır:", exact_duplicate_count)
df.loc[df.duplicated(subset=duplicate_key, keep=False)].sort_values(duplicate_key).head(6)

### Tartışma
Tekrarlı anahtar her zaman silinmesi gereken hata değildir. Aynı müşteri ve snapshot için farklı ürün satırları varsa aggregation gerekebilir. Bizim sözleşmemizde bir satır tek müşteri–snapshot olduğu için birebir tekrarları kaldıracağız. Önce gözlem birimini tanımlamadan `drop_duplicates()` çalıştırmak tehlikelidir.

In [ ]:
target_summary = (
    df["churn_30d"]
      .value_counts(dropna=False)
      .rename_axis("churn_30d")
      .to_frame("count")
)
target_summary["rate_pct"] = (target_summary["count"] / len(df) * 100).round(2)
target_summary

### Hedef dağılımını nasıl yorumlayacağım?

Accuracy'ye bakmak için henüz çok erken; model bile kurmadık. Ancak hedef oranı bize şunları düşündürür:
- Problem dengesiz mi?
- Pozitif sınıfın iş tanımı güvenilir mi?
- Etiketin oluşması için yeterli gözlem süresi var mı?
- Model çıktısına göre kaç müşteriye aksiyon alınabilir?
- Gelecek hafta accuracy yerine precision, recall ve PR-AUC neden önemli olabilir?


In [ ]:
numeric_columns = [
    "tenure_months", "monthly_fee", "avg_usage_gb_30d",
    "usage_change_90d_pct", "support_calls_90d",
    "network_incidents_30d", "late_payments_6m",
    "nps_score", "modem_age_months",
]

df[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2)

In [ ]:
def churn_rate_by(data: pd.DataFrame, column: str) -> pd.DataFrame:
    return (
        data.groupby(column, dropna=False)
            .agg(customers=("customer_id", "count"), churn_rate=("churn_30d", "mean"))
            .assign(churn_rate_pct=lambda x: (100 * x["churn_rate"]).round(2))
            .drop(columns="churn_rate")
            .sort_values("churn_rate_pct", ascending=False)
    )

print("Kontrat türüne göre:")
print(churn_rate_by(df, "contract_type").to_string())
print("\nÖdeme yöntemine göre:")
print(churn_rate_by(df, "payment_method").to_string())
print("\nŞehir grubuna göre:")
churn_rate_by(df, "city_tier")

### Burada korelasyonu nedensellik olarak yorumlamayacağım

Bir segmentte churn oranının yüksek olması, o segment bilgisinin churn'e neden olduğunu kanıtlamaz. Segmentler müşteri profili, fiyat, tenure veya başka değişkenlerle birlikte hareket ediyor olabilir. Bu tablolar hipotez üretir; tek başına nedensel karar verdirmez.

## 16. Temizleme kararlarını açık hale getirelim

Bu demo için şu kararları alacağım:
- Birebir tekrar eden satırları kaldıracağım.
- `monthly_fee` eksiklerini kontrat türü medyanıyla dolduracağım.
- `nps_score` eksikliği için hem medyan doldurma hem `nps_missing` göstergesi üreteceğim.
- Ham veriyi değiştirmeyeceğim; temiz sonucu processed katmanına yazacağım.

Bunlar evrensel doğrular değildir. Gerçek projede eksikliğin iş nedenini veri sahibiyle birlikte araştırırım.

In [ ]:
clean_df = df.drop_duplicates().copy()
clean_df["nps_missing"] = clean_df["nps_score"].isna().astype("int8")
clean_df["monthly_fee"] = clean_df["monthly_fee"].fillna(
    clean_df.groupby("contract_type")["monthly_fee"].transform("median")
)
clean_df["nps_score"] = clean_df["nps_score"].fillna(clean_df["nps_score"].median())

assert clean_df.duplicated(subset=duplicate_key).sum() == 0
assert clean_df[["monthly_fee", "nps_score"]].isna().sum().sum() == 0
assert clean_df["churn_30d"].isin([0, 1]).all()
assert clean_df["customer_id"].notna().all()

print("Temiz veri boyutu:", clean_df.shape)
print("Kalite kontrolleri başarılı.")

In [ ]:
PROCESSED_CSV_PATH = PROCESSED_DATA_DIR / "customer_snapshot_clean.csv"
PROCESSED_PARQUET_PATH = PROCESSED_DATA_DIR / "customer_snapshot_clean.parquet"

clean_df.to_csv(PROCESSED_CSV_PATH, index=False)
print(f"CSV kaydedildi: {PROCESSED_CSV_PATH}")

try:
    clean_df.to_parquet(PROCESSED_PARQUET_PATH, index=False)
    print(f"Parquet kaydedildi: {PROCESSED_PARQUET_PATH}")
except ImportError:
    print("Parquet için pyarrow/fastparquet kurulu değil; CSV çıktısıyla devam ediyorum.")

## 17. Polars ile aynı veriye bakalım (opsiyonel)

Polars kuruluysa aynı temiz CSV'yi okuyup lazy sorgu çalıştıracağım. Kurulu değilse notebook hata vermeden devam edecek. Kurulum komutu: `pip install polars`


In [ ]:
try:
    import polars as pl

    polars_summary = (
        pl.scan_csv(PROCESSED_CSV_PATH)
          .group_by("contract_type")
          .agg(
              pl.len().alias("customers"),
              pl.col("monthly_fee").mean().round(2).alias("avg_fee"),
              (100 * pl.col("churn_30d").mean()).round(2).alias("churn_rate_pct"),
          )
          .sort("churn_rate_pct", descending=True)
          .collect()
    )
    print(polars_summary)
except ImportError:
    print("Polars kurulu değil. Kavramsal karşılaştırmayı tamamladık; bu hücre opsiyoneldir.")

## 18. Feature sözleşmesi oluşturalım

Modellemeye geçmeden önce seçilen feature'ları açıkça tanımlayacağım:

| Feature | Entity | Zaman penceresi | Null kuralı | Güncelleme | Training ve serving hesabı |
|---|---|---|---|---|---|
| `avg_usage_gb_30d` | customer | Snapshot'tan önceki 30 gün | Veri yoksa 0 + flag | Günlük | Aynı ortak fonksiyon |
| `support_calls_90d` | customer | Snapshot'tan önceki 90 gün | 0 | Günlük | Aynı ortak fonksiyon |
| `monthly_fee` | customer | Snapshot anı | Kontrat medyanı | Fatura dönemi | Billing kaynağı |
| `nps_score` | customer | Snapshot'a kadar son skor | Median + missing flag | Anket sonrası | Son bilinen kayıt |
| `network_incidents_30d` | customer | Snapshot'tan önceki 30 gün | 0 | Günlük | Aynı olay filtresi |

Bu tablo küçük görünse de train–serve skew riskini ciddi biçimde azaltır.

## 19. Point-in-time correctness kontrolü

Bir müşteri için feature üretirken yalnız `event_time <= snapshot_date` kayıtlarını kullanmalıyım. Örneğin 31 Mart snapshot'ına 5 Nisan'da oluşan şikâyeti eklersem geleceği geçmişe taşımış olurum.

```python
valid_events = events.loc[events["event_time"] <= snapshot_date]
feature = valid_events.loc[
    valid_events["event_time"] > snapshot_date - pd.Timedelta(days=30)
].shape[0]
```

Bu kural yalnız train–serve skew için değil, offline değerlendirme sonuçlarının güvenilirliği için de zorunludur.

## 20. Modellemeye geçmeden önce son kontrolüm

- [ ] İş kararı ve aksiyon tanımlı mı?
- [ ] Bir satırın neyi temsil ettiği belli mi?
- [ ] Tahmin anı ve tahmin ufku belli mi?
- [ ] Hedef yalnız gelecek pencereye mi ait?
- [ ] Feature'lar tahmin anında erişilebilir mi?
- [ ] Entity ve zaman anahtarları benzersiz mi?
- [ ] Eksik, tekrar ve veri tipi kararları kayıtlı mı?
- [ ] Training ve serving feature tanımları aynı mı?
- [ ] Ham veri değişmeden korunuyor mu?
- [ ] Temiz veri yeniden üretilebilir mi?
- [ ] Basit iş kuralı/baseline tanımlı mı?

Bu soruların cevabı net değilse modelleme aşamasına geçmek için henüz erkendir.

# LAB — Problem kartı ve modellemeye hazır veri (60 dk)

## Görev 1 — Bir iş fikrini seçin (10 dk)
Kampanya, churn, çağrı hacmi, arıza veya fiber yatırım fikirlerinden birini seçin.

## Görev 2 — Problem kartını doldurun (15 dk)
- İş kararı
- Gözlem birimi
- Tahmin anı
- Hedef ve tahmin ufku
- Olası feature'lar
- Model sonrası aksiyon
- Model ve iş başarı metriği

## Görev 3 — Leakage avı (10 dk)
En az üç olası feature yazın ve tahmin anında gerçekten erişilebilir olup olmadığını açıklayın. En az bir leakage örneği bulun.

## Görev 4 — Veriyi profilleyin (15 dk)
Üretilen churn verisinde eksik değerleri, tekrarları, hedef oranını ve en az iki segmentin churn oranını inceleyin.

## Görev 5 — Feature sözleşmesi yazın (10 dk)
Üç feature için ad, entity, zaman penceresi, null kuralı, güncellenme sıklığı ve serving hesabını tanımlayın.

## 21. Haftanın sonunda elimizde ne var?

Bu haftanın sonunda yalnızca bir CSV üretmiş olmayacağız. Elimizde şunlar olacak:
- İş kararına bağlı net bir churn problem tanımı
- Açık hedef, gözlem birimi, snapshot ve tahmin ufku
- Ham ve temiz veri katmanları
- Veri sözlüğü ve feature sözleşmesi
- Eksik değer, tekrar ve hedef dağılımı analizi
- Train–serve skew ve leakage risklerinin listesi
- Bir sonraki haftanın modellemesine hazır veri seti

### Gelecek haftaya köprü
Bir sonraki hafta bu veriyi zaman bilgisine saygılı şekilde bölecek, önce basit bir baseline kuracak, ardından LightGBM ve Optuna ile ilerleyeceğiz.

### Tek cümlelik ders özeti
> Modelin kalitesi yalnız algoritmadan değil; problemin, hedefin, zamanın ve feature'ların doğru tanımlanmasından gelir.

## 22. Kapanış soruları

1. Bir iş fikrini ML problemine çevirmek için ilk sorumuz ne olmalı?
2. Gözlem birimi ve tahmin anı neden hedef değişkenden önce tanımlanmalıdır?
3. ETL ile ELT arasındaki temel fark nedir?
4. Pandas'tan Polars'a geçmek için hangi sinyallere bakarız?
5. Feature Store yalnızca bir veritabanı mıdır?
6. Train–serve skew'e iki örnek verebilir misiniz?
7. `cancellation_date` neden churn modelinde riskli bir feature'dır?
8. Point-in-time correctness sağlanmazsa offline performansa neden güvenemeyiz?